<a href="https://colab.research.google.com/github/kuz-hub/crm-cw1-sp500/blob/main/CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM & CM Coursework 1: Market Analysis and Stock Segmentation Using RFM
Study period: 1 Jan 2021 to 1 Jan 2025 (end date exclusive). Benchmark: S&P 500 index (^GSPC).

In [ ]:
!pip install yfinance -q

In [ ]:
# Import all required libraries
import os
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("yfinance", yf.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

# Task 1: Data Preparation and Feature Engineering
## 1.1 Ticker selection and data acquisition
Source of tickers: Wikipedia list of S&P 500 companies (read with pandas). Source of prices: yfinance (Yahoo Finance). I chose yfinance because it gives Close, Adjusted Close and Volume for each share in one call, as used in the Week 4 and 5 tutorials.

In [ ]:
# Part 1: Extract the S&P 500 tickers from Wikipedia using pandas
# A User-Agent is needed, otherwise Wikipedia blocks the request (HTTP 403)
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

# Yahoo uses '-' instead of '.' (BRK.B becomes BRK-B)
sp500_df['Symbol'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
tickers = sp500_df['Symbol'].tolist()

# Record the retrieval date
retrieval_date = datetime.date.today()
print('Retrieval date:', retrieval_date)
print('Total tickers extracted:', len(tickers))

sp500_df.to_csv(f'data/raw/tickers_{retrieval_date}.csv', index=False)
sp500_df[['Symbol', 'Security', 'GICS Sector']].head()

In [ ]:
# Part 2: Download daily data for all shares and the S&P 500 benchmark
# auto_adjust=False keeps both Close and Adj Close (Close is needed for Monetary)
start = '2021-01-01'
end = '2025-01-01'      # the end date is exclusive
all_symbols = tickers + ['^GSPC']

data = yf.download(all_symbols, start=start, end=end, auto_adjust=False)

adj_close = data['Adj Close']
close = data['Close']
volume = data['Volume']

print('Shapes:', adj_close.shape, close.shape, volume.shape)
print('Date range:', adj_close.index.min(), 'to', adj_close.index.max())
print('Symbols with no data at all:', adj_close.columns[adj_close.isna().all()].tolist())

In [ ]:
# Save the raw data exactly as downloaded (preserved snapshot)
adj_close.to_csv('data/raw/adj_close_daily.csv')
close.to_csv('data/raw/close_daily.csv')
volume.to_csv('data/raw/volume_daily.csv')
print(os.listdir('data/raw'))

**If Yahoo is unavailable (cached copy):** the raw files above are saved in the GitHub repository. Run the next cell *instead of* the download cells to load them. The retrieval date for the cached files is 2026-11-08.

In [ ]:
# OPTIONAL: load the cached raw data from GitHub (only if the live download fails)
# base = 'https://raw.githubusercontent.com/kuz-hub/crm-cw1-sp500/main/data/raw/'
# adj_close = pd.read_csv(base + 'adj_close_daily.csv', index_col=0, parse_dates=True)
# close = pd.read_csv(base + 'close_daily.csv', index_col=0, parse_dates=True)
# volume = pd.read_csv(base + 'volume_daily.csv', index_col=0, parse_dates=True)
# sp500_df = pd.read_csv(base + 'tickers_2026-11-08.csv')
# print(adj_close.shape)

## 1.2 Data validation and alignment
Checks: duplicate dates, missing values, non-positive prices, volume errors, unequal histories and corporate actions.

In [ ]:
# Validation checks on the raw data
print('Duplicate dates:', adj_close.index.duplicated().sum(), '| sorted:', adj_close.index.is_monotonic_increasing)

missing = adj_close.isna().sum()
print('Tickers with missing Adj Close:', (missing > 0).sum())
print(missing[missing > 0].sort_values(ascending=False))

print('Missing-value mismatch, Adj Close vs Close:', (adj_close.isna() != close.isna()).sum().sum())
print('Missing-value mismatch, Close vs Volume:', (close.isna() != volume.isna()).sum().sum())
print('Non-positive Adj Close:', (adj_close <= 0).sum().sum(), '| Non-positive Close:', (close <= 0).sum().sum())
print('Negative volume:', (volume < 0).sum().sum())

zero_volume = (volume == 0).sum()
print('Zero-volume days per ticker:')
print(zero_volume[zero_volume > 0])

In [ ]:
# Unequal histories: first valid date of each share
first_date = adj_close.apply(lambda s: s.first_valid_index())
late = first_date[first_date > adj_close.index.min()].sort_values()
print('Shares starting after', adj_close.index.min().date(), ':', len(late))
print(late)

In [ ]:
# Corporate actions: dividends and splits
# If Adj Close / Close changes over time, the share paid dividends or had adjustments
ratio = adj_close / close
print('Shares with dividend adjustments (ratio changes):', (ratio.std() > 1e-9).sum())

# A split that was not adjusted would show as a big gap between Close return and Adj Close return
close_ret = close.pct_change(fill_method=None)
adj_ret = adj_close.pct_change(fill_method=None)
print('Largest gap between Close return and Adj Close return:', round((close_ret - adj_ret).abs().max().max(), 3))

# Very large one-day moves (above 40%), to check they are real price moves
jumps = (close_ret.abs() > 0.4).sum()
print('Shares with a daily Close move above 40%:', (jumps > 0).sum())
print(jumps[jumps > 0])

**Validation rule:** a share is kept only if it has all sessions of Adj Close data in the study period and no zero-volume days. This gives every share the same period for mean return and beta, and real trading volume for RFM. Shares that fail are written to the exclusion log with a reason. Missing prices are **not** filled.

In [ ]:
# Build the exclusion log
stocks = [c for c in adj_close.columns if c != '^GSPC']
n_days = len(adj_close)
log = []

for t in stocks:
    n_obs = adj_close[t].notna().sum()
    zero_days = int((volume[t] == 0).sum())
    if n_obs == 0:
        log.append((t, 'No price data for study period', 'Excluded'))
    elif n_obs < n_days:
        log.append((t, f'Incomplete history ({n_obs} of {n_days} sessions, first valid date {adj_close[t].first_valid_index().date()})', 'Excluded'))
    elif zero_days > 0:
        log.append((t, f'Zero-volume days ({zero_days})', 'Excluded'))

exclusion_log = pd.DataFrame(log, columns=['Ticker', 'Reason', 'Action'])
valid_stocks = [t for t in stocks if t not in exclusion_log['Ticker'].tolist()]

print('Shares downloaded:', len(stocks), '| excluded:', len(exclusion_log), '| valid:', len(valid_stocks))
exclusion_log.to_csv('data/processed/exclusion_log.csv', index=False)
exclusion_log

In [ ]:
# Keep only the valid shares (plus the benchmark) in all three tables
keep = valid_stocks + ['^GSPC']
adj_close = adj_close[keep]
close = close[keep]
volume = volume[keep]

# Observation date = the last trading date that all kept series share
observation_date = adj_close.dropna().index.max()
print('Observation date:', observation_date.date())
print('Tables:', adj_close.shape, close.shape, volume.shape)

## 1.3 Weekly return and weekly beta
Weekly prices use the last available Adjusted Close in each Friday-ending week. Weekly return = this week's price / previous week's price - 1. The first and last calendar weeks are dropped if they are incomplete.

In [ ]:
# Weekly returns (same method as the Week 4/5 tutorials, but with no gap filling)
def calculate_weekly_returns(prices):
    weekly_prices = prices.resample('W-FRI').last()

    # Keep only complete calendar weeks: Monday of the week must be on/after the first date,
    # and the Friday label must be on/before the last date
    first_day = prices.index.min()
    last_day = prices.index.max()
    monday = weekly_prices.index - pd.Timedelta(days=4)
    complete = (monday >= first_day) & (weekly_prices.index <= last_day)
    weekly_prices = weekly_prices[complete]

    # pct_change = this week / previous week - 1; the first week has no previous week
    return weekly_prices.pct_change(fill_method=None).iloc[1:]

weekly_returns_all = calculate_weekly_returns(adj_close)
market_returns = weekly_returns_all['^GSPC']
weekly_returns = weekly_returns_all[valid_stocks]

print('Weekly return table:', weekly_returns.shape)
print('First week:', weekly_returns.index.min().date(), '| last week:', weekly_returns.index.max().date())
print('Missing weekly returns:', weekly_returns.isna().sum().sum())

In [ ]:
# Mean weekly return (arithmetic mean over the full period)
mean_return = weekly_returns.mean().rename('Mean_Weekly_Return')
print(mean_return.describe())

In [ ]:
# Weekly beta = covariance(share, market) / variance(market), on matched weeks (same function as the tutorial)
def calculate_weekly_beta(weekly_returns, market_returns, min_weeks=52):
    results = []
    for share in weekly_returns.columns:
        paired = pd.concat(
            [weekly_returns[share].rename('share'), market_returns.rename('market')],
            axis=1
        ).replace([np.inf, -np.inf], np.nan).dropna()

        variance = paired['market'].var(ddof=1)
        beta = np.nan
        if len(paired) >= min_weeks and variance > 0:
            beta = paired['share'].cov(paired['market']) / variance

        results.append({'Share': share, 'Beta': beta, 'Paired_weeks': len(paired)})
    return pd.DataFrame(results).set_index('Share')

beta_df = calculate_weekly_beta(weekly_returns, market_returns)
print('Aligned weekly observations used for beta:', beta_df['Paired_weeks'].unique())
print(beta_df['Beta'].describe())
beta_df.head()

In [ ]:
# Independent check of one share (AAPL) with numpy
x = weekly_returns['AAPL'].values
y = market_returns.values
check_beta = np.cov(x, y)[0, 1] / np.var(y, ddof=1)
print('AAPL beta from function:', round(beta_df.loc['AAPL', 'Beta'], 4))
print('AAPL beta from numpy check:', round(check_beta, 4))
print('AAPL mean weekly return:', round(mean_return['AAPL'], 5))

## 1.4 Adapted RFM calculations (volume based)
- **High-volume day:** volume > 2 x the mean volume of the previous 20 sessions (the day itself is excluded using `shift(1)`).
- **R (Recency):** calendar days from the latest high-volume day to the observation date (missing if no event, never 0).
- **F (Frequency):** number of high-volume days in the latest 60 sessions.
- **M (Monetary):** sum of Close x Volume over the latest 60 sessions (USD).

In [ ]:
# RFM for every valid share (same logic as the Week 3 tutorial loop)
results = []

for ticker in valid_stocks:
    vol = volume[ticker]
    price = close[ticker]

    # Mean of the previous 20 sessions, excluding the day being tested
    average_volume = vol.shift(1).rolling(20).mean()
    high_volume = vol > 2 * average_volume

    # Only days that have a full 20-session baseline can be tested
    valid_days = average_volume.notna()
    events = high_volume[valid_days & high_volume]

    # Latest 60 trading sessions ending on the observation date
    last60 = vol.index[-60:]

    if len(events) > 0:
        recency = (observation_date - events.index[-1]).days
    else:
        recency = np.nan          # no event: missing, not 0

    results.append({
        'Share': ticker,
        'Recency': recency,
        'Frequency': int(high_volume.loc[last60].sum()),
        'Monetary': float((price.loc[last60] * vol.loc[last60]).sum()),
        'History_sessions': int(valid_days.sum()) + 20
    })

rfm = pd.DataFrame(results).set_index('Share')
print('Shares with no high-volume event (Recency missing):', rfm['Recency'].isna().sum())
print(rfm.describe().round(2))
rfm.head()

### 1.5 Worked example and evidence for one share (AAPL)
A hand-style check with a plain loop, to confirm the table above is correct.

In [ ]:
# Worked example: AAPL
share = 'AAPL'
vol = volume[share]
price = close[share]
average_volume = vol.shift(1).rolling(20).mean()
high_volume = vol > 2 * average_volume

print('Observation date:', observation_date.date())
print('Last 60 sessions run from', vol.index[-60].date(), 'to', vol.index[-1].date())

# Threshold example for the last session
last_day = vol.index[-1]
print('Previous 20-session mean volume:', round(average_volume.iloc[-1]))
print('Threshold (2 x mean):', round(2 * average_volume.iloc[-1]))
print('Volume on', last_day.date(), ':', vol.iloc[-1])

# Latest high-volume day (use only days with a full 20-session baseline)
events = high_volume[average_volume.notna() & high_volume]
print('Latest high-volume day:', events.index[-1].date())
print('R =', (observation_date - events.index[-1]).days, 'calendar days')

# F and M with a plain loop over the last 60 sessions
f_count = 0
m_total = 0.0
for day in vol.index[-60:]:
    if high_volume[day]:
        f_count += 1
    m_total += price[day] * vol[day]
print('F =', f_count, '| M =', round(m_total))

print('Table values:', rfm.loc[share, ['Recency', 'Frequency', 'Monetary']].to_dict())

In [ ]:
# Edge case checks
print('Shares with at least 80 sessions of history (20 baseline + 60 window):', (rfm['History_sessions'] >= 80).sum(), 'of', len(rfm))
print('Shares with a missing Recency (no event):')
print(rfm[rfm['Recency'].isna()])
print('Shares with Frequency 0:', (rfm['Frequency'] == 0).sum())
print('Observation date has data for every kept share:', adj_close.loc[observation_date].notna().all())

In [ ]:
# Final feature table: one row per share (only shares with a valid RFM, so every clustering uses the same shares)
features = pd.concat([mean_return, beta_df[['Beta', 'Paired_weeks']], rfm[['Recency', 'Frequency', 'Monetary']]], axis=1)
features = features.join(sp500_df.set_index('Symbol')[['Security', 'GICS Sector']])

rfm_excluded = features[features['Recency'].isna()].index.tolist()
print('Shares dropped from the common set because Recency is missing:', rfm_excluded)
features = features.dropna(subset=['Mean_Weekly_Return', 'Beta', 'Recency', 'Frequency', 'Monetary'])
features.index.name = 'Share'

print('Final common share set:', len(features))
features.to_csv('data/processed/feature_table.csv')
features.describe().round(3)

In [ ]:
# Distribution plots of the features (skew and outlier check)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), ['Mean_Weekly_Return', 'Beta', 'Recency', 'Frequency', 'Monetary']):
    ax.hist(features[col], bins=30)
    ax.set_title(col)
axes.flatten()[5].hist(np.log1p(features['Monetary']), bins=30)
axes.flatten()[5].set_title('log1p(Monetary)')
plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', dpi=150)
plt.show()
print('Skewness:')
print(features[['Mean_Weekly_Return', 'Beta', 'Recency', 'Frequency', 'Monetary']].skew().round(2))